# Aadhaar oKYC probe — Sandbox.co.in

This notebook calls the **Aadhaar provider directly** (Sandbox.co.in → UIDAI oKYC), using the **exact same requests the BSS backend makes** (`api/aadhaar_client.py`). Run it to decide whether an OTP failure is on the **BSS side** or the **provider side**.

Flow (identical to BSS):
1. `POST /authenticate` (headers `x-api-key`, `x-api-secret`) → `data.access_token`
2. `POST /kyc/aadhaar/okyc/otp` (headers `x-api-key`, `Authorization: <token>`) → `data.reference_id`
3. `POST /kyc/aadhaar/okyc/otp/verify` (headers `x-api-key`, `Authorization`, `x-api-version`) → identity data

**Credentials:** set env vars `BSS_AADHAAR_API_KEY`, `BSS_AADHAAR_API_SECRET` (and optionally `BSS_AADHAAR_BASE_URL`, `BSS_AADHAAR_API_VERSION`), or paste them in the config cell. Do **not** commit real keys.

> Tip: use a real Aadhaar whose OTP you can receive. Sandbox *live* keys hit UIDAI, so it is the same system prod uses.

In [ ]:
import os, json, time, base64
import requests

BASE = os.environ.get("BSS_AADHAAR_BASE_URL", "https://api.sandbox.co.in").rstrip("/")
API_KEY = os.environ.get("BSS_AADHAAR_API_KEY", "")
API_SECRET = os.environ.get("BSS_AADHAAR_API_SECRET", "")
API_VERSION = os.environ.get("BSS_AADHAAR_API_VERSION", "1.0")

# If not using env vars, paste the live keys here instead:
# API_KEY = "key_live_..."
# API_SECRET = "secret_live_..."

assert API_KEY and API_SECRET, "Set BSS_AADHAAR_API_KEY / BSS_AADHAAR_API_SECRET (env) or paste above."
print("base   :", BASE)
print("key    :", API_KEY[:8] + "…" + API_KEY[-4:] if API_KEY else "(none)")
print("version:", API_VERSION, "| requests:", requests.__version__)

In [ ]:
def _decode_jwt_exp(token):
    try:
        p = token.split(".")[1]
        p += "=" * (-len(p) % 4)
        claims = json.loads(base64.urlsafe_b64decode(p))
        exp = claims.get("exp")
        if exp:
            print("token exp:", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(exp)),
                  "| seconds left:", int(exp - time.time()))
    except Exception as exc:
        print("token decode skipped:", exc)


# ---- Step 1: authenticate --------------------------------------------------
t0 = time.perf_counter()
try:
    r = requests.post(f"{BASE}/authenticate",
                      headers={"x-api-key": API_KEY, "x-api-secret": API_SECRET},
                      timeout=15)
    dt = time.perf_counter() - t0
    print(f"HTTP {r.status_code} in {dt:.2f}s")
    try:
        data = r.json()
    except Exception:
        data = {"raw": r.text}
    print(json.dumps(data, indent=2)[:2000])
    TOKEN = (data.get("data") or {}).get("access_token", "")
    print("token acquired:", bool(TOKEN))
    if TOKEN:
        _decode_jwt_exp(TOKEN)
except requests.RequestException as exc:
    print("NETWORK ERROR (BSS can't reach provider either if you see this?):", exc)

In [ ]:
def call(method, url, **kw):
    t0 = time.perf_counter()
    resp = getattr(requests, method)(url, **kw)
    dt = time.perf_counter() - t0
    print(f"{method.upper()} {url}\n -> HTTP {resp.status_code} in {dt:.2f}s")
    try:
        body = resp.json()
    except Exception:
        body = {"raw": resp.text}
    print(json.dumps(body, indent=2)[:3000])
    return resp, body


# ---- Step 2: generate OTP --------------------------------------------------
AADHAAR_NUMBER = "123412341234"   # <-- PUT A REAL AADHAAR (12 digits, no spaces)

gen_headers = {"x-api-key": API_KEY, "Authorization": TOKEN}
# NOTE: BSS sends this request WITHOUT x-api-version. Change to True to test if the
# provider now requires it (this is a likely 4xx cause).
INCLUDE_VERSION_ON_GENERATE = False
if INCLUDE_VERSION_ON_GENERATE:
    gen_headers["x-api-version"] = API_VERSION

r, gen = call("post", f"{BASE}/kyc/aadhaar/okyc/otp",
              headers=gen_headers,
              json={"@entity": "in.co.sandbox.kyc.aadhaar.okyc.otp.request",
                    "aadhaar_number": AADHAAR_NUMBER,
                    "consent": "y",
                    "reason": "For KYC"},
              timeout=15)

REFERENCE_ID = (gen.get("data") or {}).get("reference_id", "")
print("reference_id:", REFERENCE_ID or "(none)")

In [ ]:
# ---- Step 3: verify OTP (must run soon after generate) ---------------------
OTP = "123456"   # <-- the OTP received on the Aadhaar-registered mobile

ver_headers = {"x-api-key": API_KEY, "Authorization": TOKEN, "x-api-version": API_VERSION}
r, ver = call("post", f"{BASE}/kyc/aadhaar/okyc/otp/verify",
              headers=ver_headers,
              json={"@entity": "in.co.sandbox.kyc.aadhaar.okyc.request",
                    "reference_id": REFERENCE_ID,
                    "otp": OTP},
              timeout=20)

msg = str((ver.get("data") or {}).get("message", "")) if isinstance(ver.get("data"), dict) else ""
print("verify message:", msg)

In [ ]:
# ---- Expiry probe: reproduce 'OTP expired' --------------------------------
# Generate a fresh OTP, wait WAIT_SECONDS, then verify. This tells you the real
# provider-side validity window (UIDAI is typically ~ a few minutes).
WAIT_SECONDS = 120

_, gen2 = call("post", f"{BASE}/kyc/aadhaar/okyc/otp", headers=gen_headers,
               json={"@entity": "in.co.sandbox.kyc.aadhaar.okyc.otp.request",
                     "aadhaar_number": AADHAAR_NUMBER, "consent": "y", "reason": "For KYC"},
               timeout=15)
ref2 = (gen2.get("data") or {}).get("reference_id", "")
print(f"waiting {WAIT_SECONDS}s before verifying reference {ref2} …")
time.sleep(WAIT_SECONDS)
call("post", f"{BASE}/kyc/aadhaar/okyc/otp/verify", headers=ver_headers,
     json={"@entity": "in.co.sandbox.kyc.aadhaar.okyc.request",
           "reference_id": ref2, "otp": OTP}, timeout=20)

## How to read the results

- **`/authenticate` non-200** → provider/key problem (bad key, quota, outage). Not a BSS bug.
- **generate non-200** (e.g. 400 `x-api-version` missing, 422, 429) → provider rejects BSS's request shape/limits. Try flipping `INCLUDE_VERSION_ON_GENERATE = True`. If it then works, the BSS request is missing a required header (a BSS-side fix).
- **generate 200 but no `reference_id`** → provider response shape changed.
- **verify works here but BSS still says expired** → BSS-side bug: BSS reuses a stale `reference_id` (its `_reuse_recent` window) or waits too long. Compare the timing here vs BSS.
- **verify says `OTP expired` even right after generate** → provider/UIDAI side (the reference was already dead).
- **`429`** → rate limit on the account (wrong OTP + resend). Provider side.
- **503 / 'Source system unavailable'** → UIDAI outage. Provider side, retry later.

BSS env for reference: `BSS_AADHAAR_BASE_URL`, `BSS_AADHAAR_API_KEY`, `BSS_AADHAAR_API_SECRET`, `BSS_AADHAAR_API_VERSION`, `BSS_AADHAAR_OTP_TTL_MIN` (local validity BSS applies), `BSS_AADHAAR_MIN_INTERVAL_SEC`, `BSS_AADHAAR_GLOBAL_COOLDOWN_SEC`.